# LLM Evaluation & LLM-as-a-Judge

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/building-with-llms/08-llm-evaluation

We simulate an LLM judge with position bias and show how order-swapping debiases pairwise evaluation — and why pairwise beats pointwise.

Self-contained: NumPy + matplotlib only. No torch, no sklearn, no network, no API keys.

> **To save your work:** click **Copy to Drive** at the top, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dark style matching the site theme.
plt.style.use('dark_background')
plt.rcParams.update({
    'axes.edgecolor': '#475569',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.titlecolor': '#e2e8f0',
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'grid.color': '#2e3347',
    'savefig.facecolor': '#0f1117',
})
BRAND = '#6366f1'
TEAL = '#14b8a6'
ROSE = '#f43f5e'
YELLOW = '#eab308'

rng = np.random.default_rng(0)

## 1. A biased pairwise judge

Model A is genuinely better than B (true win prob 0.65). But our judge also has a **position bias**: it adds a fixed boost to whichever answer is shown *first*. If we always show A first, we overestimate A.

In [ ]:
def judge(true_p_A, first, position_boost=0.15, seed=0):
    """Return 'A' or 'B'. `first` is which answer is shown first."""
    g = np.random.default_rng(seed)
    p = true_p_A + (position_boost if first == 'A' else -position_boost)
    p = np.clip(p, 0, 1)
    return 'A' if g.random() < p else 'B'

# Always A-first: inflated estimate of A's win rate
N = 5000
wins_A_biased = np.mean([judge(0.65, 'A', seed=i) == 'A' for i in range(N)])
print('A win-rate, always A-first (biased):', round(wins_A_biased, 3), '(true 0.65)')

## 2. Debias by swapping order and averaging

Run each comparison twice — once with A first, once with B first — and average. The position boost cancels.

In [ ]:
def debiased_winrate(true_p_A, N=5000):
    a_first = np.mean([judge(true_p_A, 'A', seed=i) == 'A' for i in range(N)])
    b_first = np.mean([judge(true_p_A, 'B', seed=10_000+i) == 'A' for i in range(N)])
    return 0.5 * (a_first + b_first)

print('A win-rate, order-swapped (debiased):', round(debiased_winrate(0.65), 3), '(true 0.65)')

labels = ['always A-first', 'order-swapped', 'truth']
vals = [wins_A_biased, debiased_winrate(0.65), 0.65]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(labels, [v*100 for v in vals], color=[ROSE, TEAL, BRAND])
ax.axhline(65, color=YELLOW, ls='--', lw=1, alpha=0.7)
ax.set_ylabel('estimated A win-rate (%)'); ax.set_title('Order-swapping cancels position bias')
ax.grid(True, alpha=0.3, axis='y'); plt.tight_layout(); plt.show()

## 3. An eval suite score

An eval suite runs a metric over a fixed dataset and returns the mean — a single number you can gate releases on. Here a toy faithfulness check: does the answer only use facts present in the context?

In [ ]:
suite = [
    ({'context': {'paris', 'france'}, 'answer': {'paris'}}, ),
    ({'context': {'tokyo', 'japan'}, 'answer': {'tokyo', 'osaka'}}, ),  # 'osaka' not in context -> unfaithful
    ({'context': {'rome', 'italy'}, 'answer': {'rome'}}, ),
]
def faithful(ex):
    return ex['answer'].issubset(ex['context'])
score = np.mean([faithful(ex[0]) for ex in suite])
print('faithfulness suite score:', round(score, 3))

## ✏️ Your turn — debiased pairwise win-rate

Implement `debiased(a_first_rate, b_first_rate)` = the average of the two orderings' A-win-rates.

In [ ]:
def debiased(a_first_rate, b_first_rate):
    """TODO(you): return the mean of the two rates."""
    # TODO
    return ...


In [ ]:
assert abs(debiased(0.80, 0.50) - 0.65) < 1e-9
assert abs(debiased(0.60, 0.60) - 0.60) < 1e-9
print('✅ averaging both orders cancels the position boost.')

<details>
<summary>Solution</summary>

```python
def debiased(a_first_rate, b_first_rate):
    return 0.5 * (a_first_rate + b_first_rate)
```

Prefer **pairwise** over pointwise (models rank better than they grade), swap order to kill position bias, control for length, and validate the judge against human labels before trusting it.
</details>

## Recap

- LLM judges have **position, verbosity, and self-preference** biases — design around them.
- **Order-swapping and averaging** cancels position bias in pairwise evaluation.
- An **eval suite** = dataset + metric → one gateable number; run it offline (release gate) and online (sampled traffic).
- Grow the suite from real production failures so it sharpens over time.